## Setup and helpers

- `load_dotenv()` reads my keys from the `.env` file.
- `Anthropic()` makes the client. It finds `ANTHROPIC_API_KEY` by itself.
- `chat()` sends my messages to Claude and gives back only the text.
- Claude's reply is a list of blocks. I pick the text blocks by their **type**, never by position, because a thinking block can come first.
- No `temperature`: new Claude models reject it with a 400 error.
- The return line loops through `response.content` (a list of blocks), keeps only the ones where `type == "text"`, and joins their text into one string. A plain reply has one text block, but with thinking or tools there can be several blocks, so looping is safer than `content[0]`.
  - Token counts aren't blocks, so no loop needed. I read them with dot notation, like `response.usage.input_tokens` and `response.usage.output_tokens`.


In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

anthropic_client = Anthropic()
model = "claude-sonnet-5-5"


def chat(messages, system=None, tool=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if tool:
        params["tools"] = [tool]

    response = anthropic_client.messages.create(**params)

    text = "".join(block.text for block in response.content if block.type == "text")
    tool_input = next(
        (block.input for block in response.content if block.type == "tool_use"), None
    )
    return text, tool_input

In [19]:
chat([{"role": "user", "content": "say hola"}])

('¡Hola! 👋 ¿Cómo estás? Si quieres practicar español o necesitas ayuda con algo, aquí estoy.',
 None)

## Evaluate answer

- My code decides if the answer is right (`normalize_answer`). Claude only explains.
- `is_correct` picks which verdict sentence goes into the prompt.
- Claude can't see my database, so I put everything it needs into the prompt.
- It only returns one feedback sentence, so plain text is enough. No JSON needed.

**Workflow**

1. My code checks the answer with `normalize_answer` (my own function in `answer_check.py`) → True or False.
2. `is_correct` picks the verdict sentence.
3. Put the exercise, correct answer, learner's answer and verdict into the prompt.
4. Send it with `chat()`, plus the system prompt from the database.
5. Get back the feedback text and return it.


In [20]:
def evaluate_answer(
    system_prompt, exercise_prompt, correct_answer, user_answer, is_correct
):
    if is_correct:
        verdict = "The answer has already been graded as CORRECT. Explain why."
    else:
        verdict = "The answer has already been graded as INCORRECT. Explain why."

    user_prompt = f"""A learner answered this exercise.

                    Exercise: {exercise_prompt}
                    Expected answer: {correct_answer}
                    Learner's answer: {user_answer}

                    {verdict}

                    Write feedback in 1-3 short sentences, in English, addressed to the learner.
                    If the answer is wrong, explain the mistake without being discouraging.
                    If it's right, confirm it and add one short useful tip.
                    Reply with only the feedback text, nothing else."""

    messages = [{"role": "user", "content": user_prompt}]
    return chat(messages, system=system_prompt)

In [21]:
from answer_check import normalize_answer

correct_answer = "perro"
user_answer = "Pero."

is_correct = normalize_answer(user_answer) == normalize_answer(correct_answer)
print("Correct?", is_correct)

evaluate_answer(
    system_prompt="You are a friendly Spanish tutor.",
    exercise_prompt="Tengo un ___ (dog) en casa.",
    correct_answer=correct_answer,
    user_answer=user_answer,
    is_correct=is_correct,
)

Correct? False


('Close, but "pero" means "but" in Spanish. The word for "dog" is **perro**, with a double **rr** (a rolled sound). It\'s an easy one to mix up, so watch for that extra "r"!',
 None)

## Generate exercise

- I need 3 separate fields back: type, prompt, correct_answer.
- The tool schema works like a form with 3 boxes. Claude fills in each box.
- There's no real function behind this tool. I only use it for the form.
- Forcing a tool (`tool_choice`) is rejected by this model, so I ask for the tool in the prompt instead.
- I find the `tool_use` block by type, and `.input` is already a dictionary.

**Workflow**

1. Write the schema (the form with 3 boxes).
2. Send the message to Claude with `tools=[exercise_tool]`, and ask it in the prompt to use the tool.
3. Claude replies with a `tool_use` block instead of plain text.
4. Find that block by its type.
5. Read `.input` to get the dictionary with my 3 fields.


In [34]:
from ai_service import find_exercise_problem

MAX_TRIES = 3

In [35]:
exercise_tool = {
    "name": "save_exercise",
    "description": "Save one fill-in-the-blank exercise for the learner.",
    "input_schema": {
        "type": "object",
        "properties": {
            "type": {"type": "string", "description": "Always 'fill_blank'."},
            "prompt": {
                "type": "string",
                "description": "The sentence with exactly one ___ blank.",
            },
            "correct_answer": {
                "type": "string",
                "description": "The word that goes in the blank.",
            },
        },
        "required": ["type", "prompt", "correct_answer"],
    },
}

In [36]:
response = anthropic_client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write one A1 Spanish fill-in-the-blank exercise for the word 'perro' (dog). use the save_exercise tool to save it",
        }
    ],
    tools=[exercise_tool],
)

tool_use = next(block for block in response.content if block.type == "tool_use")
tool_use.input

{'type': 'fill_blank',
 'prompt': 'Mi ___ se llama Max y le gusta jugar en el parque.',
 'correct_answer': 'perro'}

In [ ]:
def generate_exercise_from_words(domain_name, system_prompt, vocabulary_items):
    target = vocabulary_items[0]
    other_words = ", ".join(item.term for item in vocabulary_items[1:]) or "none"
    level = target.cefr_level or "A1"

    user_prompt = f"""Write ONE fill-in-the-blank exercise in {domain_name} for a learner at CEFR level {level}.

                    Target word (this is the answer): {target.term} (meaning: {target.definition})
                    Optional extra words you may use: {other_words}

                    Rules:
                    - ONE short, natural sentence suitable for level {level}.
                    - Exactly ONE blank, written as ___, where the target word goes.
                    - The target word must NOT appear anywhere else in the sentence.
                    - Use the target word in exactly this form: "{target.term}".
                    - Put the English meaning in brackets right after the blank, like: ___ ({target.definition})

                    Use the save_exercise tool to save it."""

    messages = [{"role": "user", "content": user_prompt}]

    for try_number in range(1, MAX_TRIES + 1):
        _, tool_input = chat(messages, system=system_prompt, tool=exercise_tool)

        if tool_input is None:
            print(f"Try {try_number}: Claude didn't use the tool")
            continue

        problem = find_exercise_problem(tool_input, target)
        if problem is None:
            return tool_input
        print(f"Try {try_number}: rejected ({problem}): {tool_input}")

    raise ValueError(f"AI could not write a valid exercise after {MAX_TRIES} tries")

### Testing with real data from my database

- `SessionLocal()` (from my `database.py`) opens a connection to `quicklearner.db`.
- `data_manager.get_domain_by_id(db, 1)` (my function in `data_manager.py`) gets the Spanish domain row. I hard-code `1` only here for testing. In the real app, `main.py` gets `domain_id` from the request.
- `data_manager.get_vocabulary_by_domain(db, 1)[:3]` gets the words, and `[:3]` keeps the first three. The real app uses `get_due_vocabulary_items_for_user` instead, which picks words due for review.
- Printing a database object shows a memory address. Use dot notation to see values, like `domain.name` → "Spanish".
- In a notebook, only the last line of a cell shows automatically. Use `print()` for anything above it.

**Where the system prompt comes from**

1. The text lives in `data/spanish/domain.json`.
2. `seed.py` loads it into the `Domain` table (defined in `models.py`).
3. `get_domain_by_id` fetches that row.
4. `domain.system_prompt` gets passed into my function.

- To change how the tutor behaves: edit `domain.json`, then reseed.


In [45]:
from database import SessionLocal
import data_manager

db = SessionLocal()
domain = data_manager.get_domain_by_id(db, 1)
words = data_manager.get_vocabulary_by_domain(db, 1)[:3]
# print([w.term for w in words])
# print(domain.system_prompt)
generate_exercise_from_words(
    domain_name=domain.name,
    system_prompt=domain.system_prompt,
    vocabulary_items=words,
)

Try 1: got {'type': 'fill_blank', 'prompt': 'Me duele la cabeza y me duele todo el ___. Hoy no puedo ir al trabajo. (body)', 'correct_answer': 'cuerpo'}
problem: None


{'type': 'fill_blank',
 'prompt': 'Me duele la cabeza y me duele todo el ___. Hoy no puedo ir al trabajo. (body)',
 'correct_answer': 'cuerpo'}